# v135 — 全 4 Pilot モデル (`best.pt` / `last.pt`) 一括監査 & 直交アンサンブル検証 (学習ゼロ・約 6 分)

既に Drive 上で 70 epoch 完走済みの 4 手法 × 2 重み（計 8 チェックポイント）を同一 180 枚の pilot val で一括採点します：
1. **Arm A (`v127` Control)**
2. **Arm B (`v127` Spine Aux-Teacher `0.5`)** — `last.pt` で `+1.29 pt` PQ・FP `-1.15%`
3. **Small-TAL (`v130` `max_boost=4.0`)** — `last.pt` で `+1.41 pt` PQ・小個体 recall `+6.08 pt`
4. **Sibling-Shield (`v131` `sib_neg_weight=0.20`)** — `last.pt` で `+0.85 pt` PQ・単独個体 recall `+6.41 pt`

**検証する 2 つの核心命題**:
- `best.pt` 同士の比較でも `Spine-Aux`・`Small-TAL`・`Sibling-Shield` が `Arm A best.pt` を上回るか？
- `v132` で足を引っ張っていた弱い `croptrain` の代わりに、**強い直交検出器同士（`Spine-Aux + Small-TAL` 2 検出器／`+ Arm A` 3 検出器／`+ Sibling-Shield` 4 検出器）を 1-to-1 連続融合**したとき PQ・Sparse PQ (`r=0.77`) がどこまで伸びるか？


In [ ]:
# [1/3] Drive マウント・8 チェックポイント存在確認
%pip install -q "ultralytics==8.4.120" pycocotools
import hashlib, io, json, math, os, shutil, time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from pycocotools import mask as mask_utils
from tqdm.auto import tqdm
from ultralytics import YOLO

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
DATA_CACHE = DRIVE_ROOT / "competition_data"
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
PREPROC_CACHE = DRIVE_ROOT / "preprocessed_v30_q96"
LOCAL_PREPROC = Path("/content/solar_v135/preprocessed")
LOCAL_PREPROC.mkdir(parents=True, exist_ok=True)

REFINER_CKPT = DRIVE_ROOT / "runs" / "v127_spine_ab" / "refiner_edgeatt_pilot_fit365.pt"
CKPTS = {
    "arm_a": {
        "best": DRIVE_ROOT / "runs" / "v127_spine_ab" / "yolo_pilot_arm_a" / "weights" / "best.pt",
        "last": DRIVE_ROOT / "runs" / "v127_spine_ab" / "yolo_pilot_arm_a" / "weights" / "last.pt",
    },
    "spine_aux": {
        "best": DRIVE_ROOT / "runs" / "v127_spine_ab" / "yolo_pilot_arm_b" / "weights" / "best.pt",
        "last": DRIVE_ROOT / "runs" / "v127_spine_ab" / "yolo_pilot_arm_b" / "weights" / "last.pt",
    },
    "small_tal": {
        "best": DRIVE_ROOT / "runs" / "v130_small_tal" / "yolo_pilot_arm_small_tal" / "weights" / "best.pt",
        "last": DRIVE_ROOT / "runs" / "v130_small_tal" / "yolo_pilot_arm_small_tal" / "weights" / "last.pt",
    },
    "sib_shield": {
        "best": DRIVE_ROOT / "runs" / "v131_sibling_shield" / "yolo_pilot_arm_sibling_shield" / "weights" / "best.pt",
        "last": DRIVE_ROOT / "runs" / "v131_sibling_shield" / "yolo_pilot_arm_sibling_shield" / "weights" / "last.pt",
    },
}

assert REFINER_CKPT.exists(), f"Missing refiner: {REFINER_CKPT}"
for mname, d in CKPTS.items():
    for tag, p in d.items():
        assert p.exists(), f"Missing {mname}/{tag}: {p}"
print("All 8 pilot YOLO checkpoints + common Refiner verified on Drive.")


In [ ]:
# [2/3] 180 枚 Pilot 分割固定・モデル・1-to-1 連続融合モジュール
PINNED_EXCLUSIONS_CSV = """file_name,time_block_role,fold,exclusion_reason
20161229133734Ch.jpeg,development,0,within_5d_of_locked_eval
20161231133734Ch.jpeg,development,1,within_5d_of_locked_eval
20170102024454Uh.jpeg,locked_evaluation,1,within_5d_of_development
20170103030554Uh.jpeg,locked_evaluation,1,within_5d_of_development
"""

# --- BEGIN mediseg_models.py ---
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)
# --- END mediseg_models.py ---

# --- BEGIN consensus_fusion.py ---
"""1-to-1 Multi-detector bipartite matching and continuous score/mask fusion (Idea 8, v132).

Fixes three structural issues in ``run_fold_oof_eval_consensus.py`` (v120/v121c/v124c):
1. Replaces unconstrained transitive union-find with greedy descending-IoU 1-to-1 bipartite
   clustering (at most 1 proposal per detector in every cluster), preventing adjacent-filament
   chaining across detectors.
2. Replaces hard step-function vote counting (``count(s >= 0.26) >= 2``, which discards
   ``(0.45, 0.24, 0)``, ``(0.24, 0.24, 0.24)``, and solo ``(0.75, 0, 0)``) with continuous
   calibrated cluster scoring ``sum_d w_d * s_d`` (missing detector contributes ``0.0``).
3. Supports score-weighted multi-detector proposal mask fusion before ``MediSegEdgeAttUNet``.
"""


from dataclasses import dataclass

import numpy as np
from pycocotools import mask as mask_utils


@dataclass(frozen=True)
class DetectorProposal:
    det_name: str
    conf: float
    raw_rle: dict
    prop_idx: int


@dataclass
class MatchedCluster:
    members: dict[str, DetectorProposal]  # det_name -> DetectorProposal (strictly <=1 per det)

    def score_of(self, det_name: str) -> float:
        p = self.members.get(det_name)
        return float(p.conf) if p is not None else 0.0

    def weighted_score(self, weights: dict[str, float]) -> float:
        return float(sum(w * self.score_of(d) for d, w in weights.items()))

    def max_conf(self) -> float:
        return max((float(p.conf) for p in self.members.values()), default=0.0)

    def count_above(self, thr: float) -> int:
        return sum(1 for p in self.members.values() if float(p.conf) >= thr)

    def best_member(self) -> DetectorProposal:
        return max(self.members.values(), key=lambda p: (float(p.conf), p.det_name))


def match_detector_proposals_1to1(
    proposals: list[DetectorProposal],
    *,
    iou_threshold: float = 0.45,
) -> list[MatchedCluster]:
    """Group cross-detector proposals by descending mask IoU with <=1 member per detector."""
    n = len(proposals)
    if n == 0:
        return []
    rles = [p.raw_rle for p in proposals]
    iou_mat = np.asarray(mask_utils.iou(rles, rles, [0] * n), dtype=np.float64)

    parent = list(range(n))
    det_sets = [{proposals[i].det_name} for i in range(n)]

    def find(x: int) -> int:
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root

    pairs: list[tuple[float, int, int]] = []
    for i in range(n):
        di = proposals[i].det_name
        for j in range(i + 1, n):
            if di == proposals[j].det_name:
                continue
            v = float(iou_mat[i, j])
            if v >= iou_threshold:
                pairs.append((v, i, j))
    pairs.sort(key=lambda t: (-t[0], t[1], t[2]))

    for _, i, j in pairs:
        ri, rj = find(i), find(j)
        if ri != rj and det_sets[ri].isdisjoint(det_sets[rj]):
            parent[rj] = ri
            det_sets[ri] |= det_sets[rj]

    grouped: dict[int, dict[str, DetectorProposal]] = {}
    for idx in range(n):
        r = find(idx)
        p = proposals[idx]
        grouped.setdefault(r, {})[p.det_name] = p

    clusters = [MatchedCluster(members=m) for m in grouped.values()]
    clusters.sort(key=lambda c: -c.max_conf())
    return clusters


def fuse_cluster_proposal_mask(
    cluster: MatchedCluster,
    *,
    mode: str = "max_conf",
    vote_threshold: float = 0.45,
) -> np.ndarray:
    """Build bool proposal mask for ``MediSegEdgeAttUNet`` from a matched cluster."""
    if mode == "max_conf" or len(cluster.members) == 1:
        best = cluster.best_member()
        return mask_utils.decode(best.raw_rle) != 0
    if mode != "soft_vote":
        raise ValueError(f"Unknown mask fusion mode: {mode}")

    acc: np.ndarray | None = None
    w_sum = 0.0
    for p in cluster.members.values():
        w = max(float(p.conf), 1e-4)
        m = (mask_utils.decode(p.raw_rle) != 0).astype(np.float32)
        acc = (w * m) if acc is None else (acc + w * m)
        w_sum += w
    assert acc is not None
    return (acc / max(w_sum, 1e-6)) >= float(vote_threshold)
# --- END consensus_fusion.py ---

TRAIN_JSON = next(DATA_CACHE.rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
coco = json.loads(TRAIN_JSON.read_text(encoding="utf-8"))
images = {im["id"]: im for im in coco["images"]}
annotations = defaultdict(list)
for a in coco["annotations"]:
    annotations[a["image_id"]].append(a)
file_to_ids = defaultdict(list)
for iid, im in images.items():
    file_to_ids[im["file_name"]].append(iid)
train_files = sorted(file_to_ids)

fold_df = pd.read_csv(ARTIFACT_ROOT / "grouped_folds.csv")
fold_of_file = dict(zip(fold_df["file_name"], fold_df["fold"].astype(int)))
tb_path = ARTIFACT_ROOT / "time_block_assignment.csv"
if tb_path.exists():
    tb_df = pd.read_csv(tb_path)
else:
    tb_df = fold_df.copy()
    tb_df["year"] = tb_df["file_name"].str.slice(0, 4).astype(int)
    tb_df["time_block_role"] = np.where(tb_df["year"] <= 2016, "development", "locked_evaluation")
role_of_file = dict(zip(tb_df["file_name"], tb_df["time_block_role"]))
excl_path = ARTIFACT_ROOT / "spine_aux_prereg_exclusions_2026-11-13.csv"
excl_df = pd.read_csv(excl_path) if excl_path.exists() else pd.read_csv(io.StringIO(PINNED_EXCLUSIONS_CSV))
excl = set(excl_df["file_name"].tolist())
valid_files = [n for n in train_files if role_of_file[n] == "development" and fold_of_file[n] == 2 and n not in excl]
assert len(valid_files) == 180, len(valid_files)

for n in tqdm(valid_files, desc="前処理画像をローカル SSD へ"):
    lp = LOCAL_PREPROC / n
    if not lp.exists():
        shutil.copy2(PREPROC_CACHE / n, lp)

def to_rle(ann, h, w):
    seg = ann["segmentation"]
    if isinstance(seg, list):
        return mask_utils.merge(mask_utils.frPyObjects(seg, h, w))
    if isinstance(seg.get("counts"), list):
        return mask_utils.frPyObjects(seg, h, w)
    rle = dict(seg)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

gt_rles = {iid: [to_rle(a, int(im["height"]), int(im["width"])) for a in annotations[iid] if not a.get("iscrowd", 0)]
           for iid, im in images.items()}
print("Verified exact 180 pilot val observations.")


In [ ]:
# [3/3] 8 チェックポイント高速バッチ推論 + 単独比較 + 直交アンサンブル評価
def sparse_label_pq(sum_iou, tp, fp, fn, r=0.77):
    den = r * tp + 0.5 * fp + 0.5 * (1 - r) * tp + 0.5 * r * fn
    return float(r * sum_iou / den) if den > 0 else 0.0

refiner = MediSegEdgeAttUNet(3, 1, False).cuda().eval()
refiner.load_state_dict(torch.load(REFINER_CKPT, map_location="cuda", weights_only=False)["best_model"], strict=True)

def mask_square_box(mask, context=1.65, minimum=96):
    h, w = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, w, h
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = min(max(int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context)), minimum), min(h, w))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > w: x0 -= x1 - w; x1 = w
    if y1 > h: y0 -= y1 - h; y1 = h
    return max(0, x0), max(0, y0), min(w, x1), min(h, y1)

@torch.no_grad()
def refine_batch(image, proposals, batch_size=32):
    if not proposals:
        return []
    boxes = [mask_square_box(p) for p in proposals]
    crops = [cv2.resize(image[y0:y1, x0:x1], (320, 320), interpolation=cv2.INTER_LINEAR) for (x0, y0, x1, y1) in boxes]
    pa = []
    for s in range(0, len(crops), batch_size):
        ten = torch.from_numpy(np.stack([c.transpose(2, 0, 1) for c in crops[s:s + batch_size]], 0)).float().cuda() / 255.0
        pa.append(refiner(ten).sigmoid()[:, 0].cpu().numpy())
    probs = np.concatenate(pa, 0)
    out = []
    for pm, (x0, y0, x1, y1), pr in zip(proposals, boxes, probs):
        ref = np.zeros_like(pm, dtype=bool)
        ref[y0:y1, x0:x1] = cv2.resize(pr, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR) >= 0.45
        out.append(ref & pm)
    return out

def drop_contained(masks, thr=0.90):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, -1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), np.float64)
    siou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), np.float64)
    drop = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in drop or j in drop or siou[i, j] <= 0:
                continue
            inter = siou[i, j] * (areas[i] + areas[j]) / (1.0 + siou[i, j])
            sm = i if areas[i] <= areas[j] else j
            if inter / max(areas[sm], 1.0) >= thr:
                drop.add(sm)
    return [i for i in range(len(masks)) if i not in drop]

def apply_own(masks):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], np.float64)
    occ = np.zeros(masks[0].shape, bool); kept = []
    for idx in sorted(drop_contained(masks), key=lambda i: areas[i]):
        ex = masks[idx] & ~occ
        if ex.sum() >= 20:
            kept.append(ex); occ |= ex
    return kept

def cache_det(pt_path, label):
    m = YOLO(str(pt_path))
    cache = {}
    for name in tqdm(valid_files, desc=label):
        ip = LOCAL_PREPROC / name
        r = m.predict(str(ip), imgsz=1280, conf=0.08, iou=0.55, agnostic_nms=True, retina_masks=True, verbose=False)[0]
        h, w = r.orig_shape
        ms = []
        if r.masks is not None:
            for d in r.masks.data.detach().cpu().numpy():
                ms.append((cv2.resize(d, (w, h), interpolation=cv2.INTER_LINEAR) if d.shape != (h, w) else d) >= 0.5)
        cf = r.boxes.conf.detach().cpu().numpy().astype(np.float32) if r.boxes is not None else np.zeros(0, np.float32)
        vi = [i for i, x in enumerate(ms) if x.sum() >= 20]
        vms = [ms[i] for i in vi]; vcf = cf[vi] if vi else np.zeros(0, np.float32)
        im = cv2.imread(str(ip), cv2.IMREAD_COLOR)
        rms = refine_batch(im, vms)
        cache[name] = (vcf, [mask_utils.encode(np.asfortranarray(x.astype(np.uint8))) for x in vms],
                       [mask_utils.encode(np.asfortranarray(x.astype(np.uint8))) for x in rms])
    del m; torch.cuda.empty_cache()
    return cache

def score_pred_map(pred_masks_by_file):
    s_iou = 0.0; tp = 0; fp = 0; fn = 0; obs = {}
    for name in valid_files:
        preds = apply_own(pred_masks_by_file[name])
        pr = mask_utils.encode(np.asfortranarray(np.stack(preds, -1).astype(np.uint8))) if preds else []
        os_s = [0.0, 0, 0, 0]
        for iid in file_to_ids[name]:
            tr = gt_rles[iid]
            if not tr:
                fp += len(pr); os_s[2] += len(pr)
            elif not pr:
                fn += len(tr); os_s[3] += len(tr)
            else:
                mat = np.asarray(mask_utils.iou(pr, tr, [0] * len(tr)), np.float64).T
                hits = mat > 0.5
                si = float(mat[hits].sum()); ti = int(hits.sum()); fpi = int((hits.sum(0) == 0).sum()); fni = int((hits.sum(1) == 0).sum())
                s_iou += si; tp += ti; fp += fpi; fn += fni
                os_s[0] += si; os_s[1] += ti; os_s[2] += fpi; os_s[3] += fni
        obs[name] = tuple(os_s)
    den = tp + 0.5 * fp + 0.5 * fn
    return {
        "pq": float(s_iou / den),
        "pq_sparse_r077": sparse_label_pq(s_iou, tp, fp, fn, 0.77),
        "sq": float(s_iou / tp) if tp else 0.0,
        "rq": float(tp / den),
        "tp": tp, "fp": fp, "fn": fn,
    }, obs

def eval_single(cache, thr=0.26):
    pm = {n: [mask_utils.decode(cache[n][2][i]) != 0 for i in range(len(cache[n][0])) if cache[n][0][i] >= thr]
          for n in valid_files}
    return score_pred_map(pm)

def eval_weighted_fusion(caches_by_det, weights_dict, thr, discrete_vote=None):
    pm = {}
    for n in valid_files:
        props = []
        for dname, cdict in caches_by_det.items():
            cf, raw_r, _ = cdict[n]
            for i in range(len(cf)):
                props.append(DetectorProposal(dname, float(cf[i]), raw_r[i], i))
        clusters = match_detector_proposals_1to1(props, iou_threshold=0.45)
        chosen = []
        for cl in clusters:
            if discrete_vote is not None:
                ok = cl.count_above(thr) >= discrete_vote
            else:
                ok = cl.weighted_score(weights_dict) >= thr
            if ok:
                bm = cl.best_member()
                chosen.append(mask_utils.decode(caches_by_det[bm.det_name][n][2][bm.prop_idx]) != 0)
        pm[n] = chosen
    return score_pred_map(pm)

def boot(ob, oc):
    rng = np.random.default_rng(3407)
    ba = np.array([ob[f] for f in valid_files], np.float64); ca = np.array([oc[f] for f in valid_files], np.float64)
    idx = rng.integers(0, len(valid_files), size=(2000, len(valid_files)))
    bs = ba[idx].sum(1); cs = ca[idx].sum(1)
    d = cs[:, 0] / (cs[:, 1] + 0.5 * cs[:, 2] + 0.5 * cs[:, 3]) - bs[:, 0] / (bs[:, 1] + 0.5 * bs[:, 2] + 0.5 * bs[:, 3])
    return {"mean": float(d.mean()), "ci95": [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))], "p_pos": float((d > 0).mean())}

CACHES = {"best": {}, "last": {}}
for tag in ("best", "last"):
    for mname in ("arm_a", "spine_aux", "small_tal", "sib_shield"):
        CACHES[tag][mname] = cache_det(CKPTS[mname][tag], f"{mname}_{tag}")

report = {"single": {"best": {}, "last": {}}, "fusion": {"best": {}, "last": {}}}

for tag in ("best", "last"):
    print(f"\n================ SINGLE MODELS ({tag}.pt) ================")
    base_res, base_obs = eval_single(CACHES[tag]["arm_a"], 0.26)
    for mname in ("arm_a", "spine_aux", "small_tal", "sib_shield"):
        sweep = {}
        for t in (0.22, 0.24, 0.26, 0.28, 0.30):
            r, o = eval_single(CACHES[tag][mname], t)
            sweep[f"{t:.2f}"] = r
            if abs(t - 0.26) < 1e-6:
                r026, o026 = r, o
        bt = boot(base_obs, o026)
        best_t = max(sweep, key=lambda k: sweep[k]["pq"])
        report["single"][tag][mname] = {"at_026": r026, "sweep": sweep, "best_thr": best_t, "boot_vs_arm_a_026": bt}
        print(f"{mname:12s} @0.26 | PQ={r026['pq']:.5f} ({r026['pq']-base_res['pq']:+.5f}) | sparse={r026['pq_sparse_r077']:.5f} | TP={r026['tp']} FP={r026['fp']} FN={r026['fn']} | P(>0)={bt['p_pos']:.4f} | peak@{best_t}={sweep[best_t]['pq']:.5f}")

    print(f"\n================ ORTHOGONAL FUSIONS ({tag}.pt vs arm_a_{tag} @0.26) ================")
    fusions = [
        ("spine_tal_t20", {"spine_aux": 0.5, "small_tal": 0.5}, 0.20, None),
        ("spine_tal_t22", {"spine_aux": 0.5, "small_tal": 0.5}, 0.22, None),
        ("spine_tal_t24", {"spine_aux": 0.5, "small_tal": 0.5}, 0.24, None),
        ("trio_a_sp_tal_t20", {"arm_a": 1/3, "spine_aux": 1/3, "small_tal": 1/3}, 0.20, None),
        ("trio_a_sp_tal_t22", {"arm_a": 1/3, "spine_aux": 1/3, "small_tal": 1/3}, 0.22, None),
        ("cons2of3_strong_c26", {"arm_a": 1.0, "spine_aux": 1.0, "small_tal": 1.0}, 0.26, 2),
        ("quad_wmean_t21", {"small_tal": 0.35, "spine_aux": 0.35, "arm_a": 0.15, "sib_shield": 0.15}, 0.21, None),
        ("quad_wmean_t23", {"small_tal": 0.35, "spine_aux": 0.35, "arm_a": 0.15, "sib_shield": 0.15}, 0.23, None),
    ]
    for fname, wdict, thr, dvote in fusions:
        sub_c = {k: CACHES[tag][k] for k in wdict}
        fr, fo = eval_weighted_fusion(sub_c, wdict, thr, discrete_vote=dvote)
        bt = boot(base_obs, fo)
        report["fusion"][tag][fname] = {"metrics": fr, "boot_vs_arm_a_026": bt}
        print(f"{fname:21s} | PQ={fr['pq']:.5f} ({fr['pq']-base_res['pq']:+.5f}) | sparse={fr['pq_sparse_r077']:.5f} ({fr['pq_sparse_r077']-base_res['pq_sparse_r077']:+.5f}) | TP={fr['tp']} FP={fr['fp']} FN={fr['fn']} | CI95=[{bt['ci95'][0]:+.5f}, {bt['ci95'][1]:+.5f}] P(>0)={bt['p_pos']:.4f}")

mp = ARTIFACT_ROOT / "v135_pilot_4way_audit_manifest.json"
mp.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nSaved manifest:", mp)
